# Automated Model Sweep Development Notebook
This notebook trains multiple models (ResNet18, ResNet50, EfficientNet-B0) using a shared training loop function.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import tqdm
import copy
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
from utils import train_epoch, eval_epoch, set_seed

In [ ]:
# Setup Device and Seed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
set_seed(42)

In [ ]:
# Datasets and Dataloaders
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    base_transform
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    base_transform
])

batch_size = 16
train_dataset = torchvision.datasets.ImageFolder("FGVCAircraft_Subset20/train", transform=train_transform)
val_dataset = torchvision.datasets.ImageFolder("FGVCAircraft_Subset20/val", transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

num_classes = len(train_dataset.classes)
print(f"Number of classes: {num_classes}")

In [ ]:
def get_model(model_name, num_classes, freeze_backbone=False):
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
    elif model_name == 'dinov3':
        class DinoV3Classifier(nn.Module):
            def __init__(self, repo_dir, weights, num_classes, freeze_backbone=False):
                super().__init__()
                self.backbone = torch.hub.load(
                    repo_dir,
                    'dinov3_vits16',
                    source='local',
                    weights=weights
                )
                embed_dim = getattr(self.backbone, 'embed_dim', 384)
                self.fc = nn.Linear(embed_dim, num_classes)
                
                if freeze_backbone:
                    for param in self.backbone.parameters():
                        param.requires_grad = False

            def forward(self, x):
                features = self.backbone(x)
                return self.fc(features)
        
        REPO_DIR = './dinov3'
        WEIGHTS_PATH = './weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
        model = DinoV3Classifier(
            repo_dir=REPO_DIR,
            weights=WEIGHTS_PATH,
            num_classes=num_classes,
            freeze_backbone=freeze_backbone
        )
    else:
        raise ValueError(f"Model {model_name} not supported")
    return model


In [ ]:
def train_and_evaluate(model, model_name, train_loader, val_loader, device, total_epochs=32, lr=0.001):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # Initialize optimizer bound to THIS model
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_epochs)
    
    best_acc = -float("inf")
    patience = 5
    epochs_no_improve = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    
    print(f"\n{=*40}\nTraining {model_name.upper()}\n{=*40}")
    epoch_pbar = tqdm.tqdm(range(total_epochs), colour="blue")
    
    for epoch in epoch_pbar:
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, epoch, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        
        scheduler.step()
        
        # Evaluate
        val_loss, val_acc = eval_epoch(model, val_loader, criterion, epoch, device)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        epoch_pbar.write(f"Epoch [{epoch+1}/{total_epochs}] | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")
        
        # Early Stopping & Checkpointing
        if val_acc > best_acc:
            best_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"weights_{model_name}_best.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered for {model_name} at epoch {epoch+1}")
                break

In [ ]:
models_to_test = [
    {"name": "resnet18", "freeze": True, "label": "Baseline (ResNet-18 Frozen)"},
    {"name": "resnet18", "freeze": False, "label": "H1 (ResNet-18 Unfrozen)"},
    {"name": "resnet50", "freeze": False, "label": "H2 (ResNet-50 Unfrozen)"},
    {"name": "efficientnet_b0", "freeze": False, "label": "H3 (EfficientNet-B0 Unfrozen)"},
    {"name": "dinov3", "freeze": False, "label": "H4 (DINOv3 Unfrozen)"}
]

results = {}

for config in models_to_test:
    name = config["name"]
    freeze = config["freeze"]
    label = config["label"]
    
    model = get_model(name, num_classes=num_classes, freeze_backbone=freeze)
    
    best_acc, history = train_and_evaluate(
        model=model, 
        model_name=label, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        device=device,
        total_epochs=15 # Set to a lower number for quick sweeps
    )
    
    results[label] = {
        "best_val_acc": best_acc,
        "history": history
    }

print("\n--- SWEEP COMPLETE ---")
for label, res in results.items():
    print(f"{label}: Best Validation Accuracy = {res[best_val_acc]:.4f}")

import pickle
with open("sweep_results.pkl", "wb") as f:
    pickle.dump(results, f)
print("Saved results to sweep_results.pkl")
